# SimCLR pretraining: final paper figure

This notebook contains only the analysis needed to reproduce the **SimCLR pretraining panel in Figure 2A**

### Inputs

Four SimCLR training logs:

- Baseline
- Fovea-Gaze
- Periph
- Periph-NF

Note* thatthe historical internal run names are retained in the code (`periph_nonTTM`, `periph_TTM`), but the displayed labels match the final paper (`Periph`, `Periph-NF`).

### Output

`simclr_loss_and_top1_portrait.pdf`

The saved PDF contains the two stacked plots used for Figure 2A:

1. NT-Xent loss across 120 pretraining epochs
2. Contrastive Top-1 across 120 pretraining epochs

In [ ]:
import os
import re
import hashlib
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------
# Plotting style params
# ---------------------------------------------------------------------

LW_LINE = 2.0
FIGSIZE_PORTRAIT = (6.6, 7.8)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

mpl.rcParams["figure.dpi"] = 120
mpl.rcParams["savefig.dpi"] = 300
mpl.rcParams["savefig.bbox"] = "tight"
mpl.rcParams["savefig.pad_inches"] = 0.02

# Prevent the line from visually extending beyond the observed epoch range.
mpl.rcParams["axes.xmargin"] = 0.0


def set_all_font_sizes(fs=15):
    plt.rc("font", size=fs)
    plt.rc("axes", titlesize=fs)
    plt.rc("axes", labelsize=fs)
    plt.rc("xtick", labelsize=fs)
    plt.rc("ytick", labelsize=fs)
    plt.rc("legend", fontsize=fs)
    plt.rc("figure", titlesize=fs)

set_all_font_sizes(15)

In [ ]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
# Edit these four paths if your logs are stored elsewhere.
# The defaults assume the logs are placed next to this notebook.

RUNS = {
    "baseline":      Path("baseline_training.log"),
    "fovea_gaze":    Path("fovea_gaze_training.log"),
    "periph_nonTTM": Path("periph_nonTTM_training.log"),
    "periph_TTM":    Path("periph_ttm_training.log"),
}

SAVE_DIR = Path(".")
ERROR_ON_DUPLICATE = True

SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------
# Parse the SimCLR logs
# ---------------------------------------------------------------------
#   - accepts the historical log-line variants
#   - keeps the LAST occurrence for each epoch
#   - sorts epochs before plotting
#   - checks that two conditions did not accidentally point to the same log

PATS = [
    re.compile(
        r"Epoch:\s*(?P<epoch>\d+).*?"
        r"Loss:\s*(?P<loss>[-+0-9.eE]+).*?"
        r"Top1 accuracy:\s*(?P<top1>[-+0-9.eE]+)"
    ),
    re.compile(
        r"Epoch:\s*(?P<epoch>\d+).*?"
        r"Loss:\s*(?P<loss>[-+0-9.eE]+).*?"
        r"Top1:\s*(?P<top1>[-+0-9.eE]+)"
    ),
    re.compile(
        r"Epoch:\s*(?P<epoch>\d+).*?"
        r"Loss:\s*(?P<loss>[-+0-9.eE]+).*?"
        r"(Top-?1|Acc@1)\s*[:=]\s*(?P<top1>[-+0-9.eE]+)"
    ),
]


def parse_file_for_simclr(fp):
    fp = Path(fp)

    if not fp.is_file():
        raise FileNotFoundError(f"SimCLR log not found: {fp}")

    epochs, losses, top1s = [], [], []

    with fp.open("r", errors="ignore") as f:
        for line in f:
            for pat in PATS:
                m = pat.search(line)
                if m:
                    epochs.append(int(m.group("epoch")))
                    losses.append(float(m.group("loss")))
                    top1s.append(float(m.group("top1")))
                    break

    if not epochs:
        raise RuntimeError(
            f"No parsable SimCLR epochs found in {fp}. "
            "Expected lines containing Epoch, Loss, and Top1 accuracy."
        )

    uniq = {}
    for e, loss, top1 in zip(epochs, losses, top1s):
        uniq[int(e)] = (float(loss), float(top1))

    es = np.array(sorted(uniq.keys()), dtype=int)
    ls = np.array([uniq[e][0] for e in es], dtype=float)
    ts = np.array([uniq[e][1] for e in es], dtype=float)

    return es, ls, ts


def fingerprint(fp, nbytes=200_000):
    h = hashlib.sha1()
    with open(fp, "rb") as f:
        h.update(f.read(nbytes))
    return h.hexdigest()


loaded = {}
chosen_paths = {}
chosen_hashes = {}

for name, fp in RUNS.items():
    es, ls, ts = parse_file_for_simclr(fp)

    loaded[name] = (es, ls, ts)
    chosen_paths[name] = str(Path(fp).resolve())
    chosen_hashes[name] = fingerprint(fp)

    print(
        f"{name:14s} | epochs={len(es):3d} "
        f"| loss=[{np.min(ls):.4f}, {np.max(ls):.4f}] "
        f"| top1=[{np.min(ts):.2f}, {np.max(ts):.2f}]"
    )


# Duplicate-path / duplicate-content safeguard
inv_path = {}
for name, path in chosen_paths.items():
    inv_path.setdefault(path, []).append(name)

inv_hash = {}
for name, h in chosen_hashes.items():
    inv_hash.setdefault(h, []).append(name)

dup_paths = {p: names for p, names in inv_path.items() if len(names) > 1}
dup_hashes = {h: names for h, names in inv_hash.items() if len(names) > 1}

if dup_paths or dup_hashes:
    msg = ["Duplicate SimCLR log selection detected."]
    for p, names in dup_paths.items():
        msg.append(f"Same path: {p} <- {names}")
    for h, names in dup_hashes.items():
        msg.append(f"Same content hash: {h} <- {names}")

    if ERROR_ON_DUPLICATE:
        raise RuntimeError("\n".join(msg))
    else:
        print("\n".join(msg))

In [ ]:
# ---------------------------------------------------------------------
# Figure 2A: SimCLR pretraining curves
# ---------------------------------------------------------------------

COLOR = {
    "baseline":      "tab:blue",
    "fovea_gaze":    "tab:orange",
    "periph_nonTTM": "tab:green",
    "periph_TTM":    "tab:red",
}

# Display labels matching
LABEL = {
    "baseline":      "Baseline",
    "fovea_gaze":    "Fovea-Gaze",
    "periph_nonTTM": "Periph",
    "periph_TTM":    "Periph-NF",
}

ORDER = ["baseline", "fovea_gaze", "periph_nonTTM", "periph_TTM"]

OUT_PDF = SAVE_DIR / "simclr_loss_and_top1_portrait.pdf"


# Display epoch relabeling:
# logs indexed 0..119 are displayed as epochs 1..120.
def to_display_epochs(es):
    es = np.asarray(es, dtype=int)
    if es.min() == 0:
        return es + 1
    return es


XMIN, XMAX = 1.0, 120.0

fig, axes = plt.subplots(
    2,
    1,
    figsize=FIGSIZE_PORTRAIT,
    sharex=True,
)

# Top: NT-Xent loss
ax0 = axes[0]

for name in ORDER:
    es, ls, ts = loaded[name]
    x = to_display_epochs(es)

    ax0.plot(
        x,
        ls,
        color=COLOR[name],
        linewidth=LW_LINE,
        alpha=0.90,
        label=LABEL[name],
    )

ax0.set_title("SimCLR Pre-Training: NT-Xent Loss")
ax0.set_ylabel("NT-Xent loss")
ax0.grid(True, alpha=0.30)
ax0.legend(
    loc="upper right",
    frameon=True,
    framealpha=0.9,
    borderpad=0.3,
)


# Bottom: contrastive Top-1
ax1 = axes[1]

for name in ORDER:
    es, ls, ts = loaded[name]
    x = to_display_epochs(es)

    ax1.plot(
        x,
        ts,
        color=COLOR[name],
        linewidth=LW_LINE,
        alpha=0.90,
    )

ax1.set_title("SimCLR Pre-Training: Contrastive Top-1")
ax1.set_ylabel("Contrastive Top-1 (%)")
ax1.set_xlabel("Epoch")
ax1.grid(True, alpha=0.30)


# Hard clamp the x-axis
for ax in axes:
    ax.set_xlim(XMIN, XMAX)
    ax.margins(x=0)

ax1.set_xticks([1, 20, 40, 60, 80, 100, 120])

fig.tight_layout()
fig.savefig(OUT_PDF, format="pdf")

plt.show()

print(f"Wrote: {OUT_PDF}")